<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/02_Application_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# PHASE 2 - APPLICATION ANALYSIS
# CELL 1: ENVIRONMENT SETUP
# ============================================================

from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Project artifact directory
MODEL_DIR = "/content/drive/MyDrive"

print("================================")
print("PHASE 2 ENVIRONMENT")
print("================================")
print("Google Drive mounted successfully.")
print("Model directory:", MODEL_DIR)

Mounted at /content/drive
PHASE 2 ENVIRONMENT
Google Drive mounted successfully.
Model directory: /content/drive/MyDrive


In [2]:
# ============================================================
# PHASE 2 - CELL 2
# LOAD PHASE 1 ML ARTIFACTS
# ============================================================

import os
import joblib

# ------------------------------------------------------------
# Artifact paths
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    MODEL_DIR,
    "best_malware_model_lightgbm.pkl"
)

SCALER_PATH = os.path.join(
    MODEL_DIR,
    "malware_scaler.pkl"
)

ENCODER_PATH = os.path.join(
    MODEL_DIR,
    "malware_label_encoder.pkl"
)

SELECTOR_PATH = os.path.join(
    MODEL_DIR,
    "malware_variance_selector.pkl"
)

# ------------------------------------------------------------
# Verify files
# ------------------------------------------------------------

print("================================")
print("CHECKING PHASE 1 ARTIFACTS")
print("================================")

print("Model exists   :", os.path.exists(MODEL_PATH))
print("Scaler exists  :", os.path.exists(SCALER_PATH))
print("Encoder exists :", os.path.exists(ENCODER_PATH))
print("Selector exists:", os.path.exists(SELECTOR_PATH))

# ------------------------------------------------------------
# Load artifacts
# ------------------------------------------------------------

best_model = joblib.load(MODEL_PATH)
scaler = joblib.load(SCALER_PATH)
label_encoder = joblib.load(ENCODER_PATH)
selector = joblib.load(SELECTOR_PATH)

# ------------------------------------------------------------
# Display loaded components
# ------------------------------------------------------------

print("\n================================")
print("PHASE 1 ARTIFACTS LOADED")
print("================================")

print("Model          :", type(best_model).__name__)
print("Scaler         :", type(scaler).__name__)
print("Selector       :", type(selector).__name__)
print("Input features :", best_model.n_features_in_)
print("Classes        :", list(label_encoder.classes_))

CHECKING PHASE 1 ARTIFACTS
Model exists   : True
Scaler exists  : True
Encoder exists : True
Selector exists: True

PHASE 1 ARTIFACTS LOADED
Model          : LGBMClassifier
Scaler         : StandardScaler
Selector       : VarianceThreshold
Input features : 316
Classes        : ['Adware', 'Banking', 'Benign', 'Riskware', 'SMS Malware']


In [5]:
# ============================================================
# PHASE 2 - CELL 3
# APPLICATION EVIDENCE SCHEMA
# ============================================================

from pydantic import BaseModel, Field
from typing import List, Dict, Optional


class ApplicationInfo(BaseModel):
    """
    Basic information extracted from an application or APK.
    """

    app_name: Optional[str] = None
    package_name: Optional[str] = None

    version_name: Optional[str] = None
    version_code: Optional[str] = None

    description: Optional[str] = None

    permissions: List[str] = Field(default_factory=list)

    activities: List[str] = Field(default_factory=list)
    services: List[str] = Field(default_factory=list)
    receivers: List[str] = Field(default_factory=list)

    features: List[str] = Field(default_factory=list)


class VisualEvidence(BaseModel):
    """
    Evidence that will later come from screenshots/video.
    """

    extracted_text: List[str] = Field(default_factory=list)

    ui_elements: List[str] = Field(default_factory=list)

    security_indicators: List[str] = Field(default_factory=list)

    suspicious_indicators: List[str] = Field(default_factory=list)

    observed_actions: List[str] = Field(default_factory=list)


class MalwarePrediction(BaseModel):
    """
    Prediction produced by the existing Phase 1 ML model.
    """

    predicted_class: Optional[str] = None

    confidence: Optional[float] = None

    probabilities: Dict[str, float] = Field(
        default_factory=dict
    )


class SecurityEvidence(BaseModel):
    """
    Central evidence object used by the future analysis pipeline.
    """

    application: ApplicationInfo = Field(
        default_factory=ApplicationInfo
    )

    visual: VisualEvidence = Field(
        default_factory=VisualEvidence
    )

    ml_prediction: MalwarePrediction = Field(
        default_factory=MalwarePrediction
    )


print("================================")
print("APPLICATION EVIDENCE SCHEMA")
print("================================")

print("ApplicationInfo    : Ready")
print("VisualEvidence     : Ready")
print("MalwarePrediction  : Ready")
print("SecurityEvidence   : Ready")

print("\nPhase 2 schema successfully created.")

APPLICATION EVIDENCE SCHEMA
ApplicationInfo    : Ready
VisualEvidence     : Ready
MalwarePrediction  : Ready
SecurityEvidence   : Ready

Phase 2 schema successfully created.


In [6]:
# ============================================================
# PHASE 2 - CELL 4
# INSTALL APK ANALYSIS DEPENDENCY
# ============================================================

!pip -q install androguard

print("================================")
print("APK ANALYSIS DEPENDENCY")
print("================================")
print("Androguard installation completed.")


APK ANALYSIS DEPENDENCY
Androguard installation completed.


In [7]:
# ============================================================
# PHASE 2 - CELL 5
# VERIFY ANDROGUARD INSTALLATION
# ============================================================

import androguard

print("================================")
print("ANDROGUARD VERSION")
print("================================")

print("Version:", getattr(androguard, "__version__", "Unknown"))
print("Location:", androguard.__file__)

ANDROGUARD VERSION
Version: 4.1.4
Location: /usr/local/lib/python3.13/dist-packages/androguard/__init__.py


In [8]:
# ============================================================
# PHASE 2 - CELL 6
# APK METADATA EXTRACTION ENGINE
# Androguard 4.1.4
# ============================================================

from androguard.core.apk import APK


def extract_apk_metadata(apk_path: str) -> ApplicationInfo:
    """
    Extract application metadata from an APK.

    Parameters
    ----------
    apk_path : str
        Path to the APK file.

    Returns
    -------
    ApplicationInfo
        Structured application information.
    """

    # --------------------------------------------------------
    # Validate APK path
    # --------------------------------------------------------

    if not os.path.isfile(apk_path):
        raise FileNotFoundError(
            f"APK file not found: {apk_path}"
        )

    if not apk_path.lower().endswith(".apk"):
        raise ValueError(
            "Input file must have a .apk extension."
        )

    # --------------------------------------------------------
    # Parse APK
    # --------------------------------------------------------

    apk = APK(apk_path)

    # --------------------------------------------------------
    # Extract basic application information
    # --------------------------------------------------------

    app_name = apk.get_app_name()

    package_name = apk.get_package()

    version_name = apk.get_androidversion_name()

    version_code = apk.get_androidversion_code()

    # --------------------------------------------------------
    # Extract permissions
    # --------------------------------------------------------

    permissions = apk.get_permissions() or []

    # --------------------------------------------------------
    # Extract Android components
    # --------------------------------------------------------

    activities = apk.get_activities() or []

    services = apk.get_services() or []

    receivers = apk.get_receivers() or []

    # --------------------------------------------------------
    # Extract hardware/software features
    # --------------------------------------------------------

    features = []

    try:
        features = apk.get_features() or []
    except Exception:
        # Some APKs may not contain feature information.
        features = []

    # --------------------------------------------------------
    # Build structured evidence object
    # --------------------------------------------------------

    application_info = ApplicationInfo(
        app_name=app_name,
        package_name=package_name,
        version_name=version_name,
        version_code=str(version_code)
            if version_code is not None else None,
        permissions=permissions,
        activities=activities,
        services=services,
        receivers=receivers,
        features=features
    )

    return application_info


print("================================")
print("APK EXTRACTION ENGINE")
print("================================")
print("Androguard : 4.1.4")
print("Function   : extract_apk_metadata()")
print("Status     : Ready")

APK EXTRACTION ENGINE
Androguard : 4.1.4
Function   : extract_apk_metadata()
Status     : Ready


In [10]:
# ============================================================
# PHASE 2 - CELL 7
# UPLOAD APK FOR ANALYSIS
# ============================================================

from google.colab import files
import os

print("Select an APK file to analyze...")

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No APK was uploaded.")

apk_filename = next(iter(uploaded))
apk_path = os.path.abspath(apk_filename)

print("\n================================")
print("APK UPLOAD COMPLETE")
print("================================")
print("Filename:", apk_filename)
print("Path    :", apk_path)
print("Size    :", f"{os.path.getsize(apk_path) / (1024 * 1024):.2f} MB")

Select an APK file to analyze...


Saving com.iris.gallery_16.apk to com.iris.gallery_16 (2).apk

APK UPLOAD COMPLETE
Filename: com.iris.gallery_16 (2).apk
Path    : /content/com.iris.gallery_16 (2).apk
Size    : 7.93 MB


In [11]:
# ============================================================
# PHASE 2 - CELL 8
# APK STATIC ANALYSIS - DEBUG VERSION
# ============================================================

import time
from androguard.core.apk import APK

print("===============================")
print("APK STATIC ANALYSIS")
print("===============================")

print("APK:", apk_filename)
print("Starting analysis...\n")

start_time = time.time()

try:
    print("[1/3] Loading APK with Androguard...")

    apk = APK(apk_path)

    print("[2/3] APK loaded successfully.")

    print("Extracting application metadata...")

    app_info = ApplicationInfo(
        app_name=apk.get_app_name(),
        package_name=apk.get_package(),
        version_name=apk.get_androidversion_name(),
        version_code=str(
            apk.get_androidversion_code()
        ) if apk.get_androidversion_code() is not None else None,
        permissions=apk.get_permissions() or [],
        activities=apk.get_activities() or [],
        services=apk.get_services() or [],
        receivers=apk.get_receivers() or [],
        features=apk.get_features() or []
    )

    elapsed = time.time() - start_time

    print("[3/3] Metadata extraction complete.")

    print("\n===============================")
    print("APK ANALYSIS COMPLETE")
    print("===============================")

    print("Application Name :", app_info.app_name)
    print("Package Name     :", app_info.package_name)
    print("Version Name     :", app_info.version_name)
    print("Version Code     :", app_info.version_code)

    print("\nPermissions      :", len(app_info.permissions))
    print("Activities       :", len(app_info.activities))
    print("Services         :", len(app_info.services))
    print("Receivers        :", len(app_info.receivers))
    print("Features         :", len(app_info.features))

    print(f"\nAnalysis time    : {elapsed:.2f} seconds")

except Exception as e:
    print("\n===============================")
    print("APK ANALYSIS FAILED")
    print("===============================")
    print(type(e).__name__, ":", str(e))

APK STATIC ANALYSIS
APK: com.iris.gallery_16 (2).apk
Starting analysis...

[1/3] Loading APK with Androguard...
[2/3] APK loaded successfully.
Extracting application metadata...

APK ANALYSIS FAILED
AttributeError : 'str' object has no attribute 'get_app_name'


In [13]:
# ============================================================
# PHASE 2 - CELL 5
# APK METADATA PARSER
# ============================================================

!pip -q install apkutils2

print("================================")
print("APK PARSER INSTALLATION")
print("================================")
print("apkutils2 installation complete.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 819.9 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.2/201.2 kB 2.7 MB/s eta 0:00:00
APK PARSER INSTALLATION
apkutils2 installation complete.


In [15]:
# ============================================================
# PHASE 2 - CELL 6
# VERIFY APK PARSER
# ============================================================

import apkutils2
import os

print("===============================")
print("APK PARSER VERIFICATION")
print("===============================")

print("apkutils2 imported successfully.")
print("APK path      :", apk_path)
print("APK exists    :", os.path.isfile(apk_path))
print("APK size (MB) :", round(os.path.getsize(apk_path) / (1024 * 1024), 2))

APK PARSER VERIFICATION
apkutils2 imported successfully.
APK path      : /content/com.iris.gallery_16 (2).apk
APK exists    : True
APK size (MB) : 7.93


In [17]:
# ============================================================
# PHASE 2 - CELL 7
# APK MANIFEST EXTRACTION
# ============================================================

from apkutils2 import APK

print("===============================")
print("APK MANIFEST EXTRACTION")
print("===============================")

try:
    apk = APK(apk_path)

    print("APK loaded successfully.")
    print("Extracting manifest...\n")

    manifest = apk.get_manifest()

    print("===============================")
    print("MANIFEST EXTRACTION COMPLETE")
    print("===============================")

    print("Manifest type:", type(manifest))

    # Show a small preview only
    manifest_text = str(manifest)

    print("\nManifest preview:")
    print(manifest_text[:3000])

except Exception as e:
    print("\n===============================")
    print("MANIFEST EXTRACTION FAILED")
    print("===============================")
    print(type(e).__name__, ":", str(e))

APK MANIFEST EXTRACTION
APK loaded successfully.
Extracting manifest...

MANIFEST EXTRACTION COMPLETE
Manifest type: <class 'dict'>

Manifest preview:
{'@xmlns:android': 'http://schemas.android.com/apk/res/android', '@android:versionCode': '16', '@android:versionName': '0.8.0', '@android:compileSdkVersion': '35', '@android:compileSdkVersionCodename': '15', '@package': 'com.iris.gallery', '@platformBuildVersionCode': '35', '@platformBuildVersionName': '15', 'uses-sdk': {'@android:minSdkVersion': '26', '@android:targetSdkVersion': '35'}, 'uses-permission': [{'@android:name': 'android.permission.READ_MEDIA_IMAGES'}, {'@android:name': 'android.permission.READ_MEDIA_VIDEO'}, {'@android:name': 'android.permission.READ_MEDIA_VISUAL_USER_SELECTED'}, {'@android:name': 'android.permission.ACCESS_MEDIA_LOCATION'}, {'@android:name': 'android.permission.READ_EXTERNAL_STORAGE', '@android:maxSdkVersion': '32'}, {'@android:name': 'android.permission.WRITE_EXTERNAL_STORAGE', '@android:maxSdkVersion': '

In [18]:
# ============================================================
# PHASE 2 - CELL 8
# STRUCTURED APPLICATION METADATA
# ============================================================

def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]


# -----------------------------
# Basic application information
# -----------------------------

app_name = manifest.get("application", {}).get(
    "@android:label", None
)

package_name = manifest.get(
    "@package", None
)

version_name = manifest.get(
    "@android:versionName", None
)

version_code = manifest.get(
    "@android:versionCode", None
)


# -----------------------------
# Permissions
# -----------------------------

permission_entries = ensure_list(
    manifest.get("uses-permission")
)

permissions = []

for entry in permission_entries:
    if isinstance(entry, dict):
        name = entry.get("@android:name")
        if name:
            permissions.append(name)


# -----------------------------
# Application components
# -----------------------------

application = manifest.get("application", {})

activities = []
services = []
receivers = []


for entry in ensure_list(application.get("activity")):
    if isinstance(entry, dict):
        name = entry.get("@android:name")
        if name:
            activities.append(name)


for entry in ensure_list(application.get("service")):
    if isinstance(entry, dict):
        name = entry.get("@android:name")
        if name:
            services.append(name)


for entry in ensure_list(application.get("receiver")):
    if isinstance(entry, dict):
        name = entry.get("@android:name")
        if name:
            receivers.append(name)


# -----------------------------
# Build structured evidence
# -----------------------------

app_info = ApplicationInfo(
    app_name=app_name,
    package_name=package_name,
    version_name=version_name,
    version_code=version_code,
    permissions=permissions,
    activities=activities,
    services=services,
    receivers=receivers
)


# -----------------------------
# Display results
# -----------------------------

print("================================")
print("APPLICATION INFORMATION")
print("================================")

print("Package       :", app_info.package_name)
print("Version       :", app_info.version_name)
print("Version Code  :", app_info.version_code)

print("\nPermissions   :", len(app_info.permissions))
print("Activities    :", len(app_info.activities))
print("Services      :", len(app_info.services))
print("Receivers     :", len(app_info.receivers))

print("\n================================")
print("PERMISSIONS")
print("================================")

for permission in app_info.permissions:
    print("-", permission)

print("\n================================")
print("COMPONENTS")
print("================================")

print("Activities:")
for item in app_info.activities:
    print(" -", item)

print("\nServices:")
for item in app_info.services:
    print(" -", item)

print("\nReceivers:")
for item in app_info.receivers:
    print(" -", item)

APPLICATION INFORMATION
Package       : com.iris.gallery
Version       : 0.8.0
Version Code  : 16

Permissions   : 14
Activities    : 1
Services      : 0
Receivers     : 4

PERMISSIONS
- android.permission.READ_MEDIA_IMAGES
- android.permission.READ_MEDIA_VIDEO
- android.permission.READ_MEDIA_VISUAL_USER_SELECTED
- android.permission.ACCESS_MEDIA_LOCATION
- android.permission.READ_EXTERNAL_STORAGE
- android.permission.WRITE_EXTERNAL_STORAGE
- android.permission.MANAGE_EXTERNAL_STORAGE
- android.permission.POST_NOTIFICATIONS
- android.permission.RECEIVE_BOOT_COMPLETED
- android.permission.USE_BIOMETRIC
- android.permission.USE_FINGERPRINT
- android.permission.SET_WALLPAPER
- android.permission.ACCESS_NETWORK_STATE
- com.iris.gallery.DYNAMIC_RECEIVER_NOT_EXPORTED_PERMISSION

COMPONENTS
Activities:
 - com.iris.gallery.MainActivity

Services:

Receivers:
 - com.iris.gallery.MemoriesReceiver
 - com.iris.gallery.BootReceiver
 - com.iris.gallery.IrisPhotoWidget
 - androidx.profileinstaller.Pr

In [19]:
# ============================================================
# PHASE 2 - CELL 9
# SECURITY INDICATOR EXTRACTION
# ============================================================

print("================================")
print("SECURITY INDICATOR ANALYSIS")
print("================================")

security_indicators = []
suspicious_indicators = []

permissions_lower = {
    permission.lower()
    for permission in app_info.permissions
}


# ------------------------------------------------------------
# Permission-based indicators
# ------------------------------------------------------------

permission_rules = {
    "android.permission.MANAGE_EXTERNAL_STORAGE":
        "Broad filesystem/storage access",

    "android.permission.RECEIVE_BOOT_COMPLETED":
        "Can receive device boot-completed events",

    "android.permission.ACCESS_MEDIA_LOCATION":
        "Can access location metadata associated with media",

    "android.permission.POST_NOTIFICATIONS":
        "Can post notifications",

    "android.permission.SET_WALLPAPER":
        "Can set device wallpaper",

    "android.permission.USE_BIOMETRIC":
        "Can use biometric authentication",

    "android.permission.USE_FINGERPRINT":
        "Can use fingerprint authentication"
}


for permission, description in permission_rules.items():

    if permission.lower() in permissions_lower:

        security_indicators.append(
            f"{description} ({permission})"
        )


# ------------------------------------------------------------
# Potentially noteworthy permissions
# ------------------------------------------------------------

high_attention_permissions = {
    "android.permission.MANAGE_EXTERNAL_STORAGE",
    "android.permission.RECEIVE_BOOT_COMPLETED"
}


for permission in high_attention_permissions:

    if permission.lower() in permissions_lower:

        suspicious_indicators.append(
            f"Attention-required permission: {permission}"
        )


# ------------------------------------------------------------
# Component indicators
# ------------------------------------------------------------

if len(app_info.receivers) > 0:

    security_indicators.append(
        f"Application declares {len(app_info.receivers)} broadcast receiver(s)"
    )


if "com.iris.gallery.BootReceiver" in app_info.receivers:

    suspicious_indicators.append(
        "Application contains a boot-related receiver"
    )


# ------------------------------------------------------------
# Print results
# ------------------------------------------------------------

print("\nSecurity indicators:")

for item in security_indicators:
    print(" -", item)


print("\nAttention indicators:")

for item in suspicious_indicators:
    print(" -", item)


print("\n================================")
print("SECURITY INDICATOR SUMMARY")
print("================================")

print("Security indicators :", len(security_indicators))
print("Attention indicators:", len(suspicious_indicators))

SECURITY INDICATOR ANALYSIS

Security indicators:
 - Broad filesystem/storage access (android.permission.MANAGE_EXTERNAL_STORAGE)
 - Can receive device boot-completed events (android.permission.RECEIVE_BOOT_COMPLETED)
 - Can access location metadata associated with media (android.permission.ACCESS_MEDIA_LOCATION)
 - Can post notifications (android.permission.POST_NOTIFICATIONS)
 - Can set device wallpaper (android.permission.SET_WALLPAPER)
 - Can use biometric authentication (android.permission.USE_BIOMETRIC)
 - Can use fingerprint authentication (android.permission.USE_FINGERPRINT)
 - Application declares 4 broadcast receiver(s)

Attention indicators:
 - Attention-required permission: android.permission.MANAGE_EXTERNAL_STORAGE
 - Attention-required permission: android.permission.RECEIVE_BOOT_COMPLETED
 - Application contains a boot-related receiver

SECURITY INDICATOR SUMMARY
Security indicators : 8
Attention indicators: 3


In [20]:
# ============================================================
# PHASE 2 - CELL 10
# BUILD SECURITY EVIDENCE OBJECT
# ============================================================

print("================================")
print("BUILDING SECURITY EVIDENCE")
print("================================")

visual_evidence = VisualEvidence(
    security_indicators=security_indicators,
    suspicious_indicators=suspicious_indicators
)

ml_prediction = MalwarePrediction()

security_evidence = SecurityEvidence(
    application=app_info,
    visual=visual_evidence,
    ml_prediction=ml_prediction
)

print("SecurityEvidence object created successfully.")

print("\n================================")
print("EVIDENCE SUMMARY")
print("================================")

print("Package            :", security_evidence.application.package_name)
print("Version            :", security_evidence.application.version_name)
print("Permissions        :", len(security_evidence.application.permissions))
print("Activities         :", len(security_evidence.application.activities))
print("Services           :", len(security_evidence.application.services))
print("Receivers          :", len(security_evidence.application.receivers))
print("Security indicators:", len(
    security_evidence.visual.security_indicators
))
print("Attention indicators:", len(
    security_evidence.visual.suspicious_indicators
))

BUILDING SECURITY EVIDENCE
SecurityEvidence object created successfully.

EVIDENCE SUMMARY
Package            : com.iris.gallery
Version            : 0.8.0
Permissions        : 14
Activities         : 1
Services           : 0
Receivers          : 4
Security indicators: 8
Attention indicators: 3


In [21]:
# ============================================================
# PHASE 2 - CELL 11
# APK STRUCTURAL RISK FEATURES
# ============================================================

import os

print("================================")
print("APK STRUCTURAL ANALYSIS")
print("================================")

apk_size_bytes = os.path.getsize(apk_path)
apk_size_mb = apk_size_bytes / (1024 * 1024)

permission_count = len(app_info.permissions)
activity_count = len(app_info.activities)
service_count = len(app_info.services)
receiver_count = len(app_info.receivers)


# ------------------------------------------------------------
# Important permission categories
# ------------------------------------------------------------

storage_access = any(
    p in app_info.permissions
    for p in [
        "android.permission.MANAGE_EXTERNAL_STORAGE",
        "android.permission.READ_EXTERNAL_STORAGE",
        "android.permission.WRITE_EXTERNAL_STORAGE",
        "android.permission.READ_MEDIA_IMAGES",
        "android.permission.READ_MEDIA_VIDEO"
    ]
)

boot_access = (
    "android.permission.RECEIVE_BOOT_COMPLETED"
    in app_info.permissions
)

network_access = any(
    p in app_info.permissions
    for p in [
        "android.permission.INTERNET",
        "android.permission.ACCESS_NETWORK_STATE"
    ]
)

biometric_access = any(
    p in app_info.permissions
    for p in [
        "android.permission.USE_BIOMETRIC",
        "android.permission.USE_FINGERPRINT"
    ]
)


# ------------------------------------------------------------
# Structural feature dictionary
# ------------------------------------------------------------

apk_features = {
    "apk_size_mb": round(apk_size_mb, 2),
    "permission_count": permission_count,
    "activity_count": activity_count,
    "service_count": service_count,
    "receiver_count": receiver_count,
    "storage_access": storage_access,
    "boot_access": boot_access,
    "network_access": network_access,
    "biometric_access": biometric_access
}


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("APK size          :", f"{apk_size_mb:.2f} MB")
print("Permissions       :", permission_count)
print("Activities        :", activity_count)
print("Services          :", service_count)
print("Receivers         :", receiver_count)

print("\nCapability indicators:")
print("Storage access    :", storage_access)
print("Boot access       :", boot_access)
print("Network access    :", network_access)
print("Biometric access  :", biometric_access)

print("\n================================")
print("STRUCTURAL FEATURES READY")
print("================================")

print("Feature count     :", len(apk_features))

APK STRUCTURAL ANALYSIS
APK size          : 7.93 MB
Permissions       : 14
Activities        : 1
Services          : 0
Receivers         : 4

Capability indicators:
Storage access    : True
Boot access       : True
Network access    : True
Biometric access  : True

STRUCTURAL FEATURES READY
Feature count     : 9


In [22]:
# ============================================================
# PHASE 2 - CELL 12
# APK CONTENT / STRUCTURE INSPECTION
# ============================================================

import zipfile

print("================================")
print("APK CONTENT INSPECTION")
print("================================")

try:
    with zipfile.ZipFile(apk_path, "r") as apk_zip:

        files = apk_zip.namelist()

        print("Total APK entries :", len(files))

        # --------------------------------------------
        # Important APK file categories
        # --------------------------------------------

        dex_files = [
            f for f in files
            if f.endswith(".dex")
        ]

        native_libs = [
            f for f in files
            if f.startswith("lib/")
            and f.endswith((".so", ".so.gz"))
        ]

        assets = [
            f for f in files
            if f.startswith("assets/")
        ]

        resources = [
            f for f in files
            if f.startswith("res/")
        ]

        certificates = [
            f for f in files
            if f.startswith("META-INF/")
        ]

        # --------------------------------------------
        # Summary
        # --------------------------------------------

        print("\n================================")
        print("APK STRUCTURE SUMMARY")
        print("================================")

        print("DEX files          :", len(dex_files))
        print("Native libraries   :", len(native_libs))
        print("Assets             :", len(assets))
        print("Resources          :", len(resources))
        print("META-INF entries   :", len(certificates))

        print("\nDEX files:")
        for item in dex_files:
            print(" -", item)

        print("\nNative libraries:")
        for item in native_libs[:30]:
            print(" -", item)

        if len(native_libs) > 30:
            print(f" ... and {len(native_libs) - 30} more")

        # --------------------------------------------
        # Suspicious / noteworthy file patterns
        # --------------------------------------------

        interesting_files = []

        suspicious_extensions = (
            ".dex",
            ".so",
            ".sh",
            ".jar",
            ".aar"
        )

        for file_name in files:

            lower_name = file_name.lower()

            if lower_name.endswith(suspicious_extensions):
                interesting_files.append(file_name)

        print("\n================================")
        print("NOTABLE FILES")
        print("================================")

        print("Notable file count :", len(interesting_files))

        for item in interesting_files[:50]:
            print(" -", item)

        if len(interesting_files) > 50:
            print(f" ... and {len(interesting_files) - 50} more")


except Exception as e:

    print("\n================================")
    print("APK CONTENT INSPECTION FAILED")
    print("================================")

    print(type(e).__name__, ":", str(e))

APK CONTENT INSPECTION
Total APK entries : 291

APK STRUCTURE SUMMARY
DEX files          : 1
Native libraries   : 8
Assets             : 2
Resources          : 194
META-INF entries   : 73

DEX files:
 - classes.dex

Native libraries:
 - lib/arm64-v8a/libandroidx.graphics.path.so
 - lib/arm64-v8a/libavif_android.so
 - lib/armeabi-v7a/libandroidx.graphics.path.so
 - lib/armeabi-v7a/libavif_android.so
 - lib/x86/libandroidx.graphics.path.so
 - lib/x86/libavif_android.so
 - lib/x86_64/libandroidx.graphics.path.so
 - lib/x86_64/libavif_android.so

NOTABLE FILES
Notable file count : 9
 - classes.dex
 - lib/arm64-v8a/libandroidx.graphics.path.so
 - lib/arm64-v8a/libavif_android.so
 - lib/armeabi-v7a/libandroidx.graphics.path.so
 - lib/armeabi-v7a/libavif_android.so
 - lib/x86/libandroidx.graphics.path.so
 - lib/x86/libavif_android.so
 - lib/x86_64/libandroidx.graphics.path.so
 - lib/x86_64/libavif_android.so


In [23]:
# ============================================================
# PHASE 2 - CELL 13
# DEX STRUCTURAL ANALYSIS
# ============================================================

import zipfile
import tempfile
import os

print("================================")
print("DEX STRUCTURAL ANALYSIS")
print("================================")

try:
    # Extract classes.dex to a temporary location
    with zipfile.ZipFile(apk_path, "r") as apk_zip:

        dex_data = apk_zip.read("classes.dex")

    dex_path = "/content/classes.dex"

    with open(dex_path, "wb") as f:
        f.write(dex_data)

    print("classes.dex extracted.")
    print("DEX size:", round(len(dex_data) / (1024 * 1024), 2), "MB")

    # --------------------------------------------------------
    # Use androguard's DEX parser directly.
    # This avoids APK-level Androguard parsing.
    # --------------------------------------------------------

    from androguard.core.dex import DEX

    print("\nLoading DEX structure...")

    dex = DEX(dex_data)

    print("DEX loaded successfully.")

    # --------------------------------------------------------
    # Basic structural information
    # --------------------------------------------------------

    classes = list(dex.get_classes())

    print("\n================================")
    print("DEX SUMMARY")
    print("================================")

    print("Classes:", len(classes))

    # --------------------------------------------------------
    # Package/class names
    # --------------------------------------------------------

    class_names = []

    for cls in classes:
        try:
            name = cls.get_name()
            if name:
                class_names.append(name)
        except Exception:
            pass

    print("Class names extracted:", len(class_names))

    print("\nFirst 30 classes:")

    for name in class_names[:30]:
        print(" -", name)

    # --------------------------------------------------------
    # Count application-owned classes
    # --------------------------------------------------------

    app_classes = [
        name for name in class_names
        if name.startswith("Lcom/iris/gallery/")
    ]

    print("\n================================")
    print("APPLICATION CLASS SUMMARY")
    print("================================")

    print("com.iris.gallery classes:", len(app_classes))

    for name in app_classes[:30]:
        print(" -", name)

    if len(app_classes) > 30:
        print(f" ... and {len(app_classes) - 30} more")


except Exception as e:

    print("\n================================")
    print("DEX ANALYSIS FAILED")
    print("================================")

    print(type(e).__name__, ":", str(e))

DEX STRUCTURAL ANALYSIS
classes.dex extracted.
DEX size: 4.33 MB

Loading DEX structure...


2026-10-08 17:44:21.647 | DEBUG    | androguard.core.dex:__init__:8331 - DEX None None None
2026-10-08 17:44:21.649 | DEBUG    | androguard.core.dex:__init__:511 - HeaderItem
2026-10-08 17:44:21.672 | DEBUG    | androguard.core.dex:parse:7636 - Starting parsing map_item 'HEADER_ITEM'
2026-10-08 17:44:21.674 | DEBUG    | androguard.core.dex:__init__:511 - HeaderItem
2026-10-08 17:44:21.686 | DEBUG    | androguard.core.dex:parse:7757 - End of parsing map_item 'HEADER_ITEM'. Required time 0:00.0119
2026-10-08 17:44:21.687 | DEBUG    | androguard.core.dex:parse:7636 - Starting parsing map_item 'MAP_LIST'
2026-10-08 17:44:21.690 | DEBUG    | androguard.core.dex:parse:7757 - End of parsing map_item 'MAP_LIST'. Required time 0:00.0000
2026-10-08 17:44:21.692 | DEBUG    | androguard.core.dex:parse:7636 - Starting parsing map_item 'STRING_DATA_ITEM'
2026-10-08 17:44:21.753 | DEBUG    | androguard.core.dex:parse:7757 - End of parsing map_item 'STRING_DATA_ITEM'. Required time 0:00.0572
2026-10-0

DEX loaded successfully.

DEX SUMMARY
Classes: 4415
Class names extracted: 4415

First 30 classes:
 - La00;
 - Luu2;
 - Lwz0;
 - La01;
 - Lz02;
 - La02;
 - Lkv0;
 - Lsv0;
 - Le81;
 - Lmv0;
 - La03;
 - Lt30;
 - Li50;
 - Lmm;
 - Lv30;
 - Liz2;
 - La0;
 - La10;
 - La11;
 - Lea1;
 - La12;
 - Lgf;
 - La13;
 - La1;
 - La20;
 - Lep0;
 - La21;
 - La22;
 - La23;
 - La2;

APPLICATION CLASS SUMMARY
com.iris.gallery classes: 5
 - Lcom/iris/gallery/BootReceiver;
 - Lcom/iris/gallery/IrisApplication;
 - Lcom/iris/gallery/IrisPhotoWidget;
 - Lcom/iris/gallery/MainActivity;
 - Lcom/iris/gallery/MemoriesReceiver;


In [24]:
# ============================================================
# PHASE 2 - CELL 14
# APPLICATION CLASS / METHOD ANALYSIS
# ============================================================

print("================================")
print("APPLICATION METHOD ANALYSIS")
print("================================")

try:

    app_class_details = []

    for cls in classes:

        class_name = cls.get_name()

        if not class_name.startswith("Lcom/iris/gallery/"):
            continue

        methods = []

        try:
            for method in cls.get_methods():

                method_name = method.get_name()

                if method_name:
                    methods.append(method_name)

        except Exception:
            pass

        app_class_details.append({
            "class": class_name,
            "method_count": len(methods),
            "methods": sorted(set(methods))
        })


    print("Application classes:", len(app_class_details))

    for item in app_class_details:

        print("\n--------------------------------")
        print("Class:", item["class"])
        print("Methods:", item["method_count"])

        for method_name in item["methods"]:
            print(" -", method_name)


    # --------------------------------------------------------
    # Aggregate method count
    # --------------------------------------------------------

    total_app_methods = sum(
        item["method_count"]
        for item in app_class_details
    )

    print("\n================================")
    print("METHOD SUMMARY")
    print("================================")

    print("Application classes :", len(app_class_details))
    print("Application methods :", total_app_methods)


except Exception as e:

    print("\n================================")
    print("METHOD ANALYSIS FAILED")
    print("================================")

    print(type(e).__name__, ":", str(e))

APPLICATION METHOD ANALYSIS
Application classes: 5

--------------------------------
Class: Lcom/iris/gallery/BootReceiver;
Methods: 2
 - <init>
 - onReceive

--------------------------------
Class: Lcom/iris/gallery/IrisApplication;
Methods: 3
 - <init>
 - a
 - onCreate

--------------------------------
Class: Lcom/iris/gallery/IrisPhotoWidget;
Methods: 7
 - <clinit>
 - <init>
 - a
 - b
 - onAppWidgetOptionsChanged
 - onReceive
 - onUpdate

--------------------------------
Class: Lcom/iris/gallery/MainActivity;
Methods: 3
 - <init>
 - onCreate
 - onNewIntent

--------------------------------
Class: Lcom/iris/gallery/MemoriesReceiver;
Methods: 2
 - <init>
 - onReceive

METHOD SUMMARY
Application classes : 5
Application methods : 17


In [25]:
# ============================================================
# PHASE 2 - CELL 15
# SECURITY-RELEVANT API REFERENCE ANALYSIS
# ============================================================

print("================================")
print("SECURITY API REFERENCE ANALYSIS")
print("================================")

# Security-relevant API keywords
security_api_keywords = {
    "Runtime.exec": "Command execution",
    "ProcessBuilder": "Process execution",
    "DexClassLoader": "Dynamic DEX/class loading",
    "PathClassLoader": "Class loading",
    "HttpURLConnection": "HTTP network communication",
    "URLConnection": "Network communication",
    "Socket": "Low-level network communication",
    "WebView": "Embedded web content",
    "ClipboardManager": "Clipboard access",
    "PackageManager": "Application/package information access",
    "AccountManager": "Account information access",
    "LocationManager": "Location access",
    "MediaStore": "Media/storage access",
    "ContentResolver": "Content provider access",
    "NotificationManager": "Notification management",
    "AlarmManager": "Scheduled/background tasks",
    "WorkManager": "Background task scheduling",
    "BroadcastReceiver": "Broadcast event handling",
    "BootReceiver": "Boot event handling",
    "Cipher": "Cryptographic operations",
    "SecretKey": "Cryptographic key handling",
    "MessageDigest": "Hashing",
    "Base64": "Base64 encoding/decoding"
}


# ------------------------------------------------------------
# Collect strings from the DEX
# ------------------------------------------------------------

try:

    dex_strings = []

    for string in dex.get_strings():

        try:
            value = str(string)

            if value:
                dex_strings.append(value)

        except Exception:
            pass


    print("DEX strings extracted:", len(dex_strings))


    # --------------------------------------------------------
    # Search for security-related references
    # --------------------------------------------------------

    api_matches = {}

    for keyword, description in security_api_keywords.items():

        matches = [
            value
            for value in dex_strings
            if keyword.lower() in value.lower()
        ]

        if matches:
            api_matches[keyword] = {
                "description": description,
                "matches": sorted(set(matches))[:10]
            }


    # --------------------------------------------------------
    # Display results
    # --------------------------------------------------------

    print("\n================================")
    print("SECURITY API REFERENCES")
    print("================================")

    if not api_matches:

        print("No configured security API patterns found.")

    else:

        for keyword, data in api_matches.items():

            print(f"\n{keyword}")
            print("Purpose:", data["description"])

            for match in data["matches"]:
                print(" -", match)


    print("\n================================")
    print("API ANALYSIS SUMMARY")
    print("================================")

    print("Matched API categories:", len(api_matches))

except Exception as e:

    print("\n================================")
    print("API ANALYSIS FAILED")
    print("================================")

    print(type(e).__name__, ":", str(e))

SECURITY API REFERENCE ANALYSIS
DEX strings extracted: 18736

SECURITY API REFERENCES

HttpURLConnection
Purpose: HTTP network communication
 - Ljava/net/HttpURLConnection;

URLConnection
Purpose: Network communication
 - Ljava/net/HttpURLConnection;
 - Ljava/net/JarURLConnection;
 - Ljava/net/URLConnection;

Socket
Purpose: Low-level network communication
 - .OpenSSLSocketFactoryImpl
 - .OpenSSLSocketImpl
 - Failed to close timed out socket 
 - Landroid/net/ssl/SSLSockets;
 - Ljava/net/DatagramSocket;
 - Ljava/net/InetSocketAddress;
 - Ljava/net/MulticastSocket;
 - Ljava/net/Socket;
 - Ljava/net/SocketAddress;
 - Ljava/net/SocketException;

WebView
Purpose: Embedded web content
 - Landroid/webkit/WebView;

ClipboardManager
Purpose: Clipboard access
 - Landroid/content/ClipboardManager;
 - LocalClipboardManager
 - getClipboardManager

PackageManager
Purpose: Application/package information access
 - Landroid/content/pm/PackageManager$NameNotFoundException;
 - Landroid/content/pm/Packag

In [26]:
# ============================================================
# PHASE 2 - CELL 16
# APP-OWNED SECURITY API ANALYSIS
# ============================================================

print("================================")
print("APP-OWNED SECURITY API ANALYSIS")
print("================================")

security_api_keywords = {
    "HttpURLConnection": "HTTP network communication",
    "Socket": "Low-level network communication",
    "WebView": "Embedded web content",
    "ClipboardManager": "Clipboard access",
    "PackageManager": "Application/package information",
    "MediaStore": "Media/storage access",
    "ContentResolver": "Content provider access",
    "NotificationManager": "Notification management",
    "AlarmManager": "Scheduled/background tasks",
    "BroadcastReceiver": "Broadcast event handling",
    "Cipher": "Cryptographic operations",
    "MessageDigest": "Hashing",
    "Base64": "Base64 encoding/decoding",
    "Runtime": "Runtime/process operations",
    "DexClassLoader": "Dynamic code loading",
    "PathClassLoader": "Class loading"
}

app_api_matches = {}

try:

    # Inspect only classes belonging to the application
    for cls in classes:

        class_name = cls.get_name()

        if not class_name.startswith("Lcom/iris/gallery/"):
            continue

        class_matches = {}

        # Inspect methods belonging to this application class
        for method in cls.get_methods():

            method_name = method.get_name()

            try:
                method_text = str(method.get_code())

            except Exception:
                method_text = ""

            combined_text = (
                class_name + "\n" +
                method_name + "\n" +
                method_text
            )

            for keyword, description in security_api_keywords.items():

                if keyword.lower() in combined_text.lower():

                    class_matches[keyword] = description

        if class_matches:
            app_api_matches[class_name] = class_matches


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    if not app_api_matches:

        print("No configured security API references")
        print("were found directly in application methods.")

    else:

        for class_name, matches in app_api_matches.items():

            print("\n--------------------------------")
            print("Class:", class_name)

            for keyword, description in matches.items():

                print(
                    f" - {keyword}: {description}"
                )


    print("\n================================")
    print("APP API ANALYSIS SUMMARY")
    print("================================")

    print(
        "Application classes with matches:",
        len(app_api_matches)
    )

except Exception as e:

    print("\n================================")
    print("APP API ANALYSIS FAILED")
    print("================================")

    print(type(e).__name__, ":", str(e))

APP-OWNED SECURITY API ANALYSIS
No configured security API references
were found directly in application methods.

APP API ANALYSIS SUMMARY
Application classes with matches: 0


In [27]:
# ============================================================
# PHASE 2 - CELL 17
# STATIC ANALYSIS SUMMARY
# ============================================================

print("================================")
print("BUILDING STATIC ANALYSIS SUMMARY")
print("================================")

static_analysis = {
    "apk": {
        "path": apk_path,
        "size_mb": round(apk_size_mb, 2),
        "dex_files": len(dex_files),
        "native_libraries": len(native_libs),
        "assets": len(assets),
        "resources": len(resources),
        "meta_inf_entries": len(certificates)
    },

    "application": {
        "package_name": app_info.package_name,
        "version_name": app_info.version_name,
        "version_code": app_info.version_code,
        "permissions": app_info.permissions,
        "activities": app_info.activities,
        "services": app_info.services,
        "receivers": app_info.receivers
    },

    "dex": {
        "dex_size_mb": round(len(dex_data) / (1024 * 1024), 2),
        "total_classes": len(classes),
        "application_classes": len(app_class_details),
        "application_methods": total_app_methods
    },

    "security": {
        "security_indicators": security_indicators,
        "attention_indicators": suspicious_indicators,
        "broad_dex_api_categories": list(api_matches.keys()),
        "app_owned_api_categories": list(
            {
                keyword
                for matches in app_api_matches.values()
                for keyword in matches
            }
        )
    }
}


print("Static analysis summary created.")

print("\n================================")
print("STATIC ANALYSIS SUMMARY")
print("================================")

print("Package              :", static_analysis["application"]["package_name"])
print("Version              :", static_analysis["application"]["version_name"])
print("APK size             :", static_analysis["apk"]["size_mb"], "MB")

print("\nDEX")
print("Total classes        :", static_analysis["dex"]["total_classes"])
print("Application classes  :", static_analysis["dex"]["application_classes"])
print("Application methods  :", static_analysis["dex"]["application_methods"])

print("\nAPK structure")
print("DEX files            :", static_analysis["apk"]["dex_files"])
print("Native libraries     :", static_analysis["apk"]["native_libraries"])
print("Assets               :", static_analysis["apk"]["assets"])
print("Resources            :", static_analysis["apk"]["resources"])

print("\nSecurity")
print("Security indicators  :", len(
    static_analysis["security"]["security_indicators"]
))
print("Attention indicators :", len(
    static_analysis["security"]["attention_indicators"]
))
print("Broad API categories :", len(
    static_analysis["security"]["broad_dex_api_categories"]
))
print("App-owned API matches:", len(
    static_analysis["security"]["app_owned_api_categories"]
))

print("\n================================")
print("PHASE 2 STATIC ANALYSIS READY")
print("================================")

BUILDING STATIC ANALYSIS SUMMARY
Static analysis summary created.

STATIC ANALYSIS SUMMARY
Package              : com.iris.gallery
Version              : 0.8.0
APK size             : 7.93 MB

DEX
Total classes        : 4415
Application classes  : 5
Application methods  : 17

APK structure
DEX files            : 1
Native libraries     : 8
Assets               : 2
Resources            : 194

Security
Security indicators  : 8
Attention indicators : 3
Broad API categories : 15
App-owned API matches: 0

PHASE 2 STATIC ANALYSIS READY


In [28]:
# ============================================================
# PHASE 2 - CELL 18
# BUILD SECURITY EVIDENCE PACKAGE
# ============================================================

print("================================")
print("BUILDING SECURITY EVIDENCE PACKAGE")
print("================================")

# Convert static-analysis findings into a structured
# evidence package for later ML + Vision + Agentic AI stages.

security_evidence_package = {
    "application_info": app_info.model_dump(),

    "static_analysis": {
        "apk_size_mb": static_analysis["apk"]["size_mb"],
        "dex_files": static_analysis["apk"]["dex_files"],
        "native_libraries": static_analysis["apk"]["native_libraries"],
        "assets": static_analysis["apk"]["assets"],
        "resources": static_analysis["apk"]["resources"],
        "meta_inf_entries": static_analysis["apk"]["meta_inf_entries"],

        "total_dex_classes": static_analysis["dex"]["total_classes"],
        "application_classes": static_analysis["dex"]["application_classes"],
        "application_methods": static_analysis["dex"]["application_methods"],

        "broad_api_categories":
            static_analysis["security"]["broad_dex_api_categories"],

        "app_owned_api_categories":
            static_analysis["security"]["app_owned_api_categories"]
    },

    "security_indicators":
        static_analysis["security"]["security_indicators"],

    "attention_indicators":
        static_analysis["security"]["attention_indicators"],

    "ml_prediction": {
        "predicted_class": None,
        "confidence": None,
        "probabilities": {}
    },

    "visual_evidence": {
        "extracted_text": [],
        "ui_elements": [],
        "security_indicators": [],
        "suspicious_indicators": [],
        "observed_actions": []
    }
}

print("Security evidence package created.")

print("\n================================")
print("EVIDENCE PACKAGE SUMMARY")
print("================================")

print("Application package:",
      security_evidence_package["application_info"]["package_name"])

print("Security indicators:",
      len(security_evidence_package["security_indicators"]))

print("Attention indicators:",
      len(security_evidence_package["attention_indicators"]))

print("Broad API categories:",
      len(security_evidence_package["static_analysis"]["broad_api_categories"]))

print("App-owned API categories:",
      len(security_evidence_package["static_analysis"]["app_owned_api_categories"]))

print("\nML prediction slot : Ready")
print("Visual evidence slot: Ready")

print("\n================================")
print("SECURITY EVIDENCE PACKAGE READY")
print("================================")

BUILDING SECURITY EVIDENCE PACKAGE
Security evidence package created.

EVIDENCE PACKAGE SUMMARY
Application package: com.iris.gallery
Security indicators: 8
Attention indicators: 3
Broad API categories: 15
App-owned API categories: 0

ML prediction slot : Ready
Visual evidence slot: Ready

SECURITY EVIDENCE PACKAGE READY
